In [9]:
import math
from datetime import date

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font

# --------------------------------------------------
# 設定値（ここを書き換えれば内容を変更できます）
# --------------------------------------------------
FONT_NAME = "MS PGothic"          # 日本語フォント
INVOICE_NO = "0001"               # 請求書番号（先頭0を保持するため文字列）
COMPANY = "株式会社ABC"
ADDRESS = "〒101-0022 東京都千代田区神田練塀町300"
TEL_FAX = "TEL:03-1234-5678 FAX:03-1234-5678"
PERSON = "担当者名:鈴木一郎 様"
TAX_RATE = 0.1                    # 消費税率 10%

# True  : 金額・小計・消費税・合計を「数式」で書き込む（Excelで開けば自動計算）
# False : Python側で計算した「数値」を直接書き込む
#         → Excel以外のビューア（プレビュー、pandas等）でも必ず値が表示される
USE_FORMULAS = True

# 明細（商品名, 数量, 単価）
ITEMS = [
    ("商品A", 2, 10000),
    ("商品B", 1, 15000),
]

today = date.today()
FILENAME = f"請求書_{today:%Y%m%d}.xlsx"


def main() -> None:
    wb = Workbook()
    ws = wb.active
    ws.title = "請求書"

    base_font = Font(name=FONT_NAME, size=11)
    bold_font = Font(name=FONT_NAME, size=11, bold=True)
    title_font = Font(name=FONT_NAME, size=16, bold=True)
    right = Alignment(horizontal="right")

    # ---------- 列幅 ----------
    for col, width in {"A": 3, "B": 26, "C": 10, "D": 12,
                       "E": 14, "F": 8, "G": 13}.items():
        ws.column_dimensions[col].width = width

    # ---------- タイトル ----------
    ws["B2"] = "請求書"
    ws["B2"].font = title_font

    # ---------- 宛先情報 ----------
    ws["B4"] = COMPANY
    ws["B5"] = ADDRESS
    ws["B6"] = TEL_FAX
    ws["B7"] = PERSON

    # ---------- 請求書番号・日付 ----------
    ws["F4"] = "No."
    ws["G4"] = INVOICE_NO
    ws["G4"].number_format = "@"          # 文字列扱い（0001 の先頭0を残す）

    ws["F5"] = "日付"
    ws["G5"] = today                       # 現在日付
    ws["G5"].number_format = "yyyy/mm/dd"

    # ---------- 明細表のヘッダー ----------
    for i, header in enumerate(["商品名", "数量", "単価", "金額"]):
        ws.cell(row=10, column=2 + i, value=header).font = bold_font

    # ---------- 明細行 ----------
    first_row = 11
    last_row = first_row + len(ITEMS) - 1
    amounts = []
    for offset, (name, qty, price) in enumerate(ITEMS):
        r = first_row + offset
        amount = qty * price
        amounts.append(amount)
        ws.cell(row=r, column=2, value=name)
        ws.cell(row=r, column=3, value=qty)
        ws.cell(row=r, column=4, value=price)
        # 金額 = 数量 × 単価
        ws.cell(row=r, column=5,
                value=f"=C{r}*D{r}" if USE_FORMULAS else amount)

    # ---------- 明細合計・小計・消費税・合計 ----------
    total_row = last_row + 1               # 13行目
    subtotal_row = total_row + 2           # 15行目
    tax_row = subtotal_row + 1             # 16行目
    grand_row = subtotal_row + 2           # 17行目
    rate_row = grand_row + 2               # 19行目（消費税率）

    subtotal = sum(amounts)
    tax = math.floor(subtotal * TAX_RATE)  # 端数切り捨て
    grand_total = subtotal + tax

    ws.cell(row=total_row, column=5,
            value=f"=SUM(E{first_row}:E{last_row})" if USE_FORMULAS else subtotal)

    ws.cell(row=subtotal_row, column=2, value="小計")
    ws.cell(row=subtotal_row, column=5,
            value=f"=E{total_row}" if USE_FORMULAS else subtotal)

    ws.cell(row=tax_row, column=2, value="消費税")
    ws.cell(row=tax_row, column=5,
            value=f"=ROUNDDOWN(E{subtotal_row}*$C${rate_row},0)"
            if USE_FORMULAS else tax)

    ws.cell(row=grand_row, column=2, value="合計")
    ws.cell(row=grand_row, column=5,
            value=f"=E{subtotal_row}+E{tax_row}" if USE_FORMULAS else grand_total)

    # ---------- 消費税率（前提条件を明示するセル） ----------
    ws.cell(row=rate_row, column=2, value="※消費税率（変更する場合は右のセルを修正）")
    rate_cell = ws.cell(row=rate_row, column=3, value=TAX_RATE)
    rate_cell.number_format = "0%"

    # ---------- フォントと配置の統一（罫線なし） ----------
    for row in ws.iter_rows(min_row=1, max_row=rate_row, min_col=1, max_col=7):
        for cell in row:
            if cell.value is None:
                continue
            if cell.coordinate != "B2":
                cell.font = bold_font if cell.font.bold else base_font
            if isinstance(cell.value, (int, float)) or (
                isinstance(cell.value, str) and cell.value.startswith("=")
            ):
                cell.alignment = right

    wb.save(FILENAME)
    print(f"作成しました: {FILENAME}  （小計 {subtotal} / 消費税 {tax} / 合計 {grand_total}）")


if __name__ == "__main__":
    main()

作成しました: 請求書_20260816.xlsx  （小計 35000 / 消費税 3500 / 合計 38500）
